In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
food = os.path.join(path,'Q1_data.csv')
df = pd.read_csv(food)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=10, edgecolor='red')
plt.title('delivery_time')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task 1: Write your code here:
df.drop("Order_ID",inplace=True,axis=1)


In [ ]:
# Task 2: Write your code here:

# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
#completion of task2
import numpy as np

cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
  df[col].fillna(df[col].mode()[0])



In [ ]:
# Task 3: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder

label_encoder = LabelEncoder()

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = label_encoder.fit_transform(df[col])

In [ ]:
# Task 5: Write your code here: split x and y
X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)




In [ ]:
# Task 6: Write your code here:
# Target imbalance means the classes in your target variable are NOT evenly distributed - one class appears WAY more than the other.
# Count each class
print(y.value_counts())

# Show as percentages
print(y.value_counts(normalize=True) * 100)

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split, StratifiedKFold
X_train, X_test, y_train, y_test = X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
model = RandomForestRegressor()

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"MAE : {np.mean(mae_scores):.2f}")
print("-"*40)




In [ ]:
X.info()

In [ ]:
# Task 1: Write your code here:
# Feature importance

feature_cols = ["Distance_km","Weather","Traffic_Level","Time_of_Day","Vehicle_Type","Preparation_Time_min","Courier_Experience_yrs"]
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Histogram
plt.hist(y_pred, bins=30)
plt.xlabel("time")
plt.ylabel("Frequency")
plt.title("Histogram")
plt.show()

In [ ]:
# Task Bonus: Write your code here: